# Disaster Tweet Classification with TF-IDF and Logistic Regression  

In this notebook, we tackle the **Disaster Tweets classification problem** using a **TF-IDF representation** of text and a **Logistic Regression classifier**.a **simple TF-IDF + Logistic Regression pipeline** can be surprisingly effective when tuned properly.

- TF-IDF transforms raw tweets into numerical vectors that capture the relative importance of words.  
- Logistic Regression provides a strong, interpretable linear classifier with just one key parameter (C) to control regularization.  
- By experimenting with different values of C and max_features in TF-IDF, we identify the **best configuration** that balances performance and generalization.  

This simplicity has clear advantages:  
- Faster training and evaluation compared to deep models.  
- Lower risk of overfitting, since the model is constrained.  



In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Table of Contents
- [1. Data Loading](#data-loading)
- [2. Text Cleaning](#text-cleaning)
- [3. Train-Test Split](#train-test-split)
- [4. Hyperparameter Ranges](#hyperparameter-ranges)
- [5. Model Training & Validation](#model-training--validation)
- [6. Cross-Validation](#cross-validation)
- [7. Evaluation Metrics](#evaluation-metrics)
- [8. Confusion Matrix](#confusion-matrix)
- [9. Results Summary](#results-summary)

---

# 1. Data Loading
We load the dataset (`train.csv` and `test.csv`) provided in the Kaggle competition.  
- `X`: tweets in text form (features).  
- `y`: labels (0 = not disaster, 1 = disaster).  

---

# 2. Text Cleaning
A **custom transformer (TextCleaner)** is created and integrated into the ML pipeline.  

Cleaning steps:  
- Lowercasing text → standardizes representation.  
- Removing mentions, hashtags, URLs, punctuation, and numbers → reduces noise.  
- Tokenization → splits text into individual words.  
- Lemmatization → reduces words to their base form (e.g., “fires” → “fire”), reducing dimensionality.  
- Stopword removal → removes uninformative words (e.g., “the”, “and”) that don’t add predictive value.   

---

# 3. Train-Test Split
We split data into **training set (80%)** and **validation set (20%)** using stratified sampling.  

- **Stratification:** ensures class proportions (disaster vs non-disaster) are preserved in both train and validation sets. For example, if 40% of tweets are disasters, stratification guarantees ~40% disasters in each split.  

**Why?**  
- Prevents imbalance between classes in the validation set.  
- Gives a fairer estimate of real-world performance.  

---

# 4. Hyperparameter Ranges
We experiment with:  
- **C (Logistic Regression regularization strength):** [1.0, 1.2, 1.4, 1.6, 2.0].  
  - Smaller C → stronger regularization → simpler model (less overfitting risk).  
  - Larger C → weaker regularization → more complex model (may capture subtle patterns, but risk overfitting).  
- **max_features (TF-IDF vocabulary size):** [3000, 6000, 6800, 8000, 10000].  
  - Smaller vocab → less information, risk of underfitting.  
  - Larger vocab → captures more nuances but may include noise.  

**Why?** Exploring multiple values helps us find the “sweet spot” where the model captures enough useful features without overfitting.  

---

# 5. Model Training & Validation
We build a **Pipeline** with:  
1. TextCleaner (custom preprocessing).  
2. TfidfVectorizer (converts text into numerical vectors based on importance of words).  
3. LogisticRegression (classification model).  

For each `(C, max_features)` pair:  
- Train model on training data.  
- Predict on validation data.  
- Record metrics: F1, Accuracy, Precision, Recall, Confusion Matrix.  
  

---

# 6. Cross-Validation
We run **3-fold cross-validation** on the entire dataset.  
- Data is split into 3 parts → each fold is used once for validation while others are used for training.  
- Scores are averaged to compute **Mean CV F1**.  

**Why?**  
- A single validation split may be biased by randomness.  
- Cross-validation provides a more reliable estimate of performance across different subsets of the data.  

---

# 7. Evaluation Metrics
We use multiple metrics to evaluate models:  

- **Accuracy:**  
  - Formula: `(TP + TN) / (TP + TN + FP + FN)`  
  - Meaning: Proportion of correctly classified tweets.  
  - **Why?** If 90% of tweets are “not disaster,” a model predicting all tweets as “not disaster” gets 90% accuracy but is completely useless for detecting disasters. 

- **Precision (Positive Predictive Value):**  
  - Formula: `TP / (TP + FP)`  
  - Meaning: Of tweets predicted as disasters, how many were correct?  
  - **Why?** Important when false alarms (FP) are costly. For example, falsely flagging normal tweets as disaster-related could create panic.  

- **Recall (Sensitivity or True Positive Rate):**  
  - Formula: `TP / (TP + FN)`  
  - Meaning: Of all actual disasters, how many did we catch?  
  - **Why?** Important when missing true disasters (FN) is dangerous.  

- **F1 Score (Harmonic Mean of Precision and Recall):**  
  - Formula: `2 * (Precision * Recall) / (Precision + Recall)`  
  - Meaning: Balances precision and recall in one number.  
  - **Why?** Kaggle uses F1 as the main competition metric, since it handles imbalanced data better than accuracy.  

**Takeaway:**  
- Use **F1 Score** for model selection (competition metric).  
- Check **precision vs recall trade-off** to ensure the model is robust for real-world use.  

---

# 8. Confusion Matrix
The confusion matrix breaks predictions into four categories:  

|                | Predicted: No Disaster | Predicted: Disaster |
|----------------|-------------------------|---------------------|
| **Actual: No** | TN (True Negatives)     | FP (False Positives) |
| **Actual: Yes**| FN (False Negatives)    | TP (True Positives) |

- **TN (True Negatives):** Correctly predicted non-disasters.  
- **TP (True Positives):** Correctly predicted disasters.  
- **FP (False Positives):** Mistakenly predicted disasters (false alarm).  
- **FN (False Negatives):** Missed disasters (critical misses).  

**Why?**  
- Helps us see *where* the model fails.  
- Example: High FN → model is missing many real disasters (bad recall). High FP → model is over-alerting (bad precision).  

---

# 9. Results Summary
We compile results into a table of `(C, max_features)` combinations with metrics.  
- Sorted by **Validation F1** (primary Kaggle metric) and then **Mean CV F1** (stability).  

**Why?**  
- This allows quick comparison and helps select the best hyperparameters.  
- Ensures we don’t pick a model that only looks good on one split but fails in generalization.  


In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score,precision_score, recall_score

# ---- Load Data ----
train_df = pd.read_csv("../input/nlp-getting-started/train.csv")
test_df  = pd.read_csv("../input/nlp-getting-started/test.csv")

X = train_df["text"]
y = train_df["target"]

# ---- Custom Cleaner ----
class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return [self.clean_text(str(text)) for text in X]
    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r"(@[A-Za-z0-9_]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|^rt|http.+?", " ", text)
        text = re.sub(r"\d+", " ", text)  # remove numbers
        tokens = nltk.word_tokenize(text)
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens if t not in self.stop_words]
        return " ".join(tokens)

# ---- Train-Test Split ----
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---- Hyperparameter ranges ----
C_values = [1.0, 1.2, 1.4, 1.6, 2.0]
max_features_values = [3000, 6000,6800, 8000,10000]

results = []

serial = 1
for C in C_values:
    for mf in max_features_values:
        # ---- Pipeline ----
        pipe = Pipeline([
            ("cleaner", TextCleaner()),
            ("tfidf", TfidfVectorizer(max_features=mf, ngram_range=(1,1), sublinear_tf=True)),
            ("clf", LogisticRegression(C=C, max_iter=1000, solver="liblinear"))
        ])

        # ---- Train ----
        pipe.fit(X_train, y_train)

        # ---- Validation ----
        y_pred = pipe.predict(X_val)
        val_f1 = f1_score(y_val, y_pred, average="macro")
        acc = (y_val == y_pred).mean()

        # ---- CV ----
        scores = cross_val_score(pipe, X, y, cv=3, scoring="f1")
        mean_cv_f1 = scores.mean()

        #---P&R ---
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        # ---- Confusion matrix ----
        cm = confusion_matrix(y_val, y_pred)
        tn, fp, fn, tp = cm.ravel()

        results.append({
            "S.No": serial,
            "C": C,
            "max_features": mf,
            "Validation F1": val_f1,
            "Mean CV F1": mean_cv_f1,
            "Accuracy": acc,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "precision": precision,
            "recall":recall
        })
        serial += 1

# ---- Results Summary ----
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Validation F1", "Mean CV F1"], ascending=False).reset_index(drop=True)

print("Validation & CV Metrics with Confusion Matrices:\n")
print(results_df.to_string(index=False))


In [ ]:
# ---- Final Training & Submission (Best Config) ----
best_pipe = Pipeline([
    ("cleaner", TextCleaner()),
    ("tfidf", TfidfVectorizer(max_features=6800, ngram_range=(1,1), sublinear_tf=True)),
    ("clf", LogisticRegression(C=1.4, max_iter=1000, solver="liblinear"))
])

# Train on full training data
best_pipe.fit(X, y)

# Predict on test set
test_preds = best_pipe.predict(test_df["text"])

# Build submission DataFrame
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

# Save to CSV
submission.to_csv("submission.csv", index=False)
print("Submission file created: submission.csv")
